# Kalshi Manual Trader

`buy_limit(coin, qty, limit, side, window, on_fill)`

In [ ]:
import asyncio
import json
import uuid
from datetime import datetime, timedelta, timezone

import httpx
from kalshi_auth import KALSHI_BASE, KALSHI_WS, sign_headers

ET_OFFSET = timedelta(hours=-4)

SERIES = {
    "BTC": "KXBTC15M",
    "ETH": "KXETH15M",
    "SOL": "KXSOL15M",
    "XRP": "KXXRP15M",
}

print("Auth ready")

In [ ]:
# ── Ticker Resolution ────────────────────────────────────────────────────────

def _ticker_from_close(series: str, close_utc: datetime) -> str:
    close_et = close_utc + ET_OFFSET
    return series + "-" + close_et.strftime("%y%b%d%H%M").upper() + "-" + close_et.strftime("%M")


def _get_window_bounds(which: str) -> tuple[datetime, datetime]:
    """Return (open_utc, close_utc) for 'current' or 'next' 15-min window."""
    now_et    = datetime.now(timezone.utc) + ET_OFFSET
    mins      = (now_et.minute // 15) * 15
    cur_open  = now_et.replace(minute=mins, second=0, microsecond=0)
    cur_close = cur_open + timedelta(minutes=15)

    if which == "next":
        cur_open  += timedelta(minutes=15)
        cur_close += timedelta(minutes=15)

    return cur_open - ET_OFFSET, cur_close - ET_OFFSET   # return as UTC


def resolve_ticker(coin: str, window: str = "current") -> str:
    """Resolve the Kalshi ticker for a coin in current or next window."""
    coin = coin.upper()
    series = SERIES[coin]
    _, close_utc = _get_window_bounds(window)
    ticker = _ticker_from_close(series, close_utc)
    open_utc, _ = _get_window_bounds(window)
    open_et  = open_utc + ET_OFFSET
    close_et = close_utc + ET_OFFSET
    print(f"{coin} {window} window: {ticker}  ({open_et.strftime('%H:%M')}–{close_et.strftime('%H:%M ET')})")
    return ticker

# quick test
resolve_ticker("BTC", "current")
resolve_ticker("BTC", "next")

In [ ]:
# ── Order Primitives ──────────────────────────────────────────────────────────

_client = httpx.AsyncClient()

async def _place_order(ticker: str, action: str, side: str, qty: int, limit: float) -> dict | None:
    """Place a limit order. Returns the full order dict or None on failure."""
    yes_price = f"{limit:.2f}" if side == "yes" else f"{1.0 - limit:.2f}"
    order = {
        "ticker":            ticker,
        "action":            action,
        "side":              side,
        "count":             qty,
        "type":              "limit",
        "yes_price_dollars": yes_price,
        "client_order_id":   str(uuid.uuid4()),
    }
    r = await _client.post(
        f"{KALSHI_BASE}/portfolio/orders",
        headers=sign_headers("POST", "/trade-api/v2/portfolio/orders"),
        content=json.dumps(order),
        timeout=8,
    )
    if r.status_code in (200, 201):
        od = r.json().get("order", {})
        print(f"  {action.upper()} {side.upper()} {qty}ct @ {limit:.2f}  oid={od.get('order_id','')[:8]}")
        return od
    else:
        print(f"  FAILED {r.status_code}: {r.text[:200]}")
        return None


async def _cancel_order(oid: str) -> bool:
    r = await _client.delete(
        f"{KALSHI_BASE}/portfolio/orders/{oid}",
        headers=sign_headers("DELETE", f"/trade-api/v2/portfolio/orders/{oid}"),
        timeout=8,
    )
    if r.status_code in (200, 204):
        print(f"  Cancelled {oid[:8]}")
        return True
    elif r.status_code == 404:
        print(f"  Already gone {oid[:8]}")
        return True
    else:
        print(f"  Cancel failed {r.status_code}: {r.text[:120]}")
        return False


async def _wait_fill_then_sell(oid: str, ticker: str, side: str, qty: int, sell_limit: float):
    """Poll order status; once filled, place a sell at sell_limit."""
    print(f"  Watching {oid[:8]} for fill → will sell @ {sell_limit:.2f}")
    while True:
        await asyncio.sleep(3)
        r = await _client.get(
            f"{KALSHI_BASE}/portfolio/orders/{oid}",
            headers=sign_headers("GET", f"/trade-api/v2/portfolio/orders/{oid}"),
            timeout=8,
        )
        if r.status_code != 200:
            continue
        status = r.json().get("order", {}).get("status", "")
        remaining = int(r.json().get("order", {}).get("remaining_count_fp", qty))
        filled = qty - remaining
        if filled > 0:
            ts = datetime.now().strftime("%H:%M:%S")
            print(f"  [{ts}] FILLED {filled}ct — placing SELL {side.upper()} @ {sell_limit:.2f}")
            await _place_order(ticker, "sell", side, filled, sell_limit)
            return
        if status in ("canceled", "cancelled"):
            print(f"  Order {oid[:8]} was cancelled — no sell placed")
            return

print("Order primitives ready")

In [ ]:
# ── Main API ─────────────────────────────────────────────────────────────────

async def buy_limit(
    coin: str,
    qty: int,
    limit: float,
    side: str = "yes",
    window: str = "current",
    on_fill: float | None = None,
):
    """
    Place a limit buy on Kalshi.

    Args:
        coin:    "BTC", "ETH", "SOL", "XRP"
        qty:     number of contracts
        limit:   price in dollars (0.01–0.99)
        side:    "yes" or "no"
        window:  "current" or "next"
        on_fill: if set, once the buy fills place a sell at this price
    """
    ticker = resolve_ticker(coin, window)
    od = await _place_order(ticker, "buy", side, qty, limit)
    if od and on_fill is not None:
        oid = od.get("order_id", "")
        asyncio.create_task(_wait_fill_then_sell(oid, ticker, side, qty, on_fill))
    return od


async def cancel(oid: str):
    """Cancel an order by ID (full or prefix — but full is safer)."""
    return await _cancel_order(oid)


async def my_orders(coin: str | None = None, status: str = "resting"):
    """List open/resting orders, optionally filtered by coin."""
    params = {"status": status, "limit": 100}
    if coin:
        series = SERIES[coin.upper()]
        params["ticker"] = series
    r = await _client.get(
        f"{KALSHI_BASE}/portfolio/orders",
        headers=sign_headers("GET", "/trade-api/v2/portfolio/orders"),
        params=params,
        timeout=8,
    )
    if r.status_code == 200:
        orders = r.json().get("orders", [])
        for o in orders:
            side  = o.get("side", "?")
            price = o.get("yes_price_dollars") or o.get("yes_price", "?")
            rem   = o.get("remaining_count_fp", "?")
            print(f"  {o['ticker']}  {o['action']} {side.upper()} {rem}ct @ {price}  status={o['status']}  oid={o['order_id'][:8]}")
        print(f"  ({len(orders)} orders)")
        return orders
    else:
        print(f"  Failed {r.status_code}: {r.text[:200]}")
        return []


async def cancel_all(coin: str | None = None):
    """Cancel all resting orders, optionally filtered by coin."""
    orders = await my_orders(coin, status="resting")
    for o in orders:
        await _cancel_order(o["order_id"])
    print(f"  Cancelled {len(orders)} orders")

print("API ready: buy_limit, cancel, my_orders, cancel_all")

## Usage Examples

In [ ]:
# Buy YES in current window
await buy_limit("BTC", qty=5, limit=0.44, side="yes")

In [ ]:
# Buy YES in next window
await buy_limit("BTC", qty=5, limit=0.44, side="yes", window="next")

In [ ]:
# Buy YES in current window, on fill sell at 0.78
await buy_limit("BTC", qty=5, limit=0.44, side="yes", on_fill=0.78)

In [ ]:
# List resting orders
await my_orders()

In [ ]:
# Cancel all resting orders
# await cancel_all()

# Cancel all for a specific coin
# await cancel_all("BTC")